# Violence Against Nursing-Facility Caregivers: What OSHA's 2025 Case Data Shows

Reproduces every figure in [`README.md`](README.md) from OSHA's Injury Tracking Application (ITA) case detail data, and the state and size tables in [`../data/`](../data/).

**Author** Yuxuan Huang · Vantara Medical Equipment (Mahoraga Vantara LLC), New York
**Data year** 2025 (submissions through 2026-03-15)
**Source data** US Government work, 17 U.S.C. §105

---
## 1. Environment

In [ ]:
!pip install -q duckdb pandas requests
import os, re, duckdb, requests, pandas as pd, numpy as np
pd.set_option('display.max_columns', None); pd.set_option('display.width', 200)
os.makedirs('data', exist_ok=True); os.makedirs('derived', exist_ok=True); os.makedirs('../data', exist_ok=True)
con = duckdb.connect(':memory:')

---
## 2. Download

OSHA publishes ITA files at [osha.gov/Establishment-Specific-Injury-and-Illness-Data](https://www.osha.gov/Establishment-Specific-Injury-and-Illness-Data). The file name changes with each release; update the URL below from that page when OSHA republishes. OSHA's server returns HTTP 403 without a browser User-Agent header.

In [ ]:
CASES_URL = 'https://www.osha.gov/sites/default/largefiles/ITA_Case_Detail_Data_2025_through_3-15-2026.csv'
HDRS = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/120.0 Safari/537.36'}
PATH = 'data/osha_cases.csv'
if not os.path.exists(PATH):
    with requests.get(CASES_URL, headers=HDRS, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(PATH, 'wb') as f:
            for chunk in r.iter_content(1 << 20): f.write(chunk)
print('downloaded', round(os.path.getsize(PATH) / 1e6), 'MB')

---
## 3. Load nursing-facility cases (NAICS 6231)

Read verbatim as text. Column names are resolved case-insensitively because OSHA has renamed fields between releases. Day counts use `-9999` as a sentinel; negative values are set to null.

In [ ]:
con.execute(f'''CREATE OR REPLACE TABLE raw_cases AS
    SELECT * FROM read_csv_auto('{PATH}', header=true, all_varchar=true, ignore_errors=true, sample_size=-1)''')
cols = [r[0] for r in con.execute('DESCRIBE raw_cases').fetchall()]
print(con.execute('SELECT COUNT(*) FROM raw_cases').fetchone()[0], 'rows, all industries')

def col(*names):
    low = {c.lower(): c for c in cols}
    for n in names:
        if n.lower() in low: return low[n.lower()]
    raise KeyError(names)
F = dict(naics=col('naics_code'), state=col('state'), est=col('establishment_ID'), size=col('size'),
         job=col('job_description'), soc=col('SOC_Description'),
         dafw=col('dafw_num_away'), djtr=col('djtr_num_tr'),
         before=col('New_nar_before_incident'), what=col('New_nar_what_happened'),
         injury=col('New_nar_injury_illness'), obj=col('New_nar_object_substance'))

df = con.execute(f'''
SELECT "{F['state']}" AS state, "{F['est']}" AS est, "{F['size']}" AS size,
       "{F['job']}" AS job, "{F['soc']}" AS soc,
       CASE WHEN TRY_CAST("{F['dafw']}" AS INT) >= 0 THEN TRY_CAST("{F['dafw']}" AS INT) END AS dafw,
       CASE WHEN TRY_CAST("{F['djtr']}" AS INT) >= 0 THEN TRY_CAST("{F['djtr']}" AS INT) END AS djtr,
       lower(concat_ws(' ', "{F['before']}", "{F['what']}", "{F['injury']}", "{F['obj']}")) AS narr
FROM raw_cases WHERE "{F['naics']}" LIKE '6231%'
''').df()
print(f'nursing-facility cases: {len(df):,}')

---
## 4. Classification rules

**Violence.** A named person (resident, patient, client, visitor, family member) followed within a few words by a hostile act; or the act described passively as done *by* such a person; or combative / aggressive behavior together with a hostile act; or an assault by a co-worker. Contact where the employee struck an object ("hit her head on the cabinet") is excluded unless aggression is also described. Resistance during care (pulling back, pushing back) is not counted as violence on its own.

**Resident handling.** Identical to the frozen rule in [`../nursing-facility-handling/`](../nursing-facility-handling/), so that the two modules' counts are consistent.

In [ ]:
PERSON = r"(?:resident|residents|patient|patients|pt|res|client|member|visitor|family member)"
HOSTILE = (r"(?:hit|hits|struck|strikes?|punch\w*|kick\w*|bit|bites?|biting|bitten|scratch\w*|slap\w*|spit|spat|"
           r"head[- ]?butt\w*|pinch\w*|squeez\w*|dug|swung at|wrapped (?:her |his |their )?arms|"
           r"grabb?(?:ed|ing|s)? (?:the )?(?:employee|ee|staff|her|his|my|onto|arms?|forearm|hand|wrist|rt|lt|right|left)\b|"
           r"twist(?:ed|ing) (?:the )?(?:employee|ee|her|his|my)\b(?! (?:right |left )?(?:knee|ankle|back|foot))|push(?:ed|ing) (?:the )?(?:employee|ee|staff|her|him)\b(?! back)|"
           r"spray\w*|assault\w*|attack\w*)")
A = rf"\b{PERSON}\b(?:\W+\w+){{0,5}}?\W+{HOSTILE}"
PASSIVE = rf"\b(?:bit|bitten|hit|struck|kicked|punched|scratched|attacked|assaulted|grabbed)\b(?:\W+\w+){{0,4}}?\W+by (?:a |the )?{PERSON}\b"
AGG = r"\b(?:combative|aggressive|aggression|agitated|assault\w*|attacked|human bite|physical aggression|resident attack)\b"
AGG_ACT = r"\b(?:pull\w*|grab\w*|punch\w*|hit|kick\w*|bit|bite|scratch\w*|twist\w*|push\w*|swung|squeez\w*)\b"
COWORKER = r"\b(?:co-?workers?|employees?|staff members?)\b(?:\W+\w+){0,3}?\W+(?:jumped|attacked|assaulted|punched|sprayed)\b"
ACC = (r"\b(?:hit|struck|bumped|banged)\b (?:her |his |my |their |the )?(?:left |right |l |r )?"
       r"(?:head|hand|knee|arm|elbow|shin|foot|toe|big toe|leg|shoulder|finger)\b(?:\W+\w+){0,3}?\W+(?:on|against|into)\b")

def is_violence(s):
    s = s or ''
    if re.search(COWORKER, s): return True
    if re.search(A, s) or re.search(PASSIVE, s):
        return not (re.search(ACC, s) and not re.search(AGG, s))
    if re.search(AGG, s) and re.search(AGG_ACT, s): return True
    return False

# resident handling: frozen strict rule from nursing-facility-handling
PERSON_H = r"\b(?:resident|residents|patient|patients|pt|res)\b"
MOVEMENT = (r"\b(?:transfer\w*|lift\w*|reposition\w*|boost\w*|turn\w*|roll\w*|"
            r"pull\w* (?:up|over)|pivot\w*|hoyer|mechanical lift|gait belt|sit[- ]to[- ]stand|"
            r"slide sheet|draw sheet|bed to chair|chair to bed|wheelchair to bed|bed to wheelchair|"
            r"toilet\w* transfer|ambulat\w*)\b")
PHRASE = r"patient handling|resident handling|resident care transfer"

n = df.narr.fillna('')
df['violence'] = n.map(is_violence)
df['handling'] = (n.str.contains(PERSON_H, regex=True) & n.str.contains(MOVEMENT, regex=True)) | n.str.contains(PHRASE, regex=True)
df['violence_during_handling'] = df.violence & df.handling

with open('derived/classification-rule.txt', 'w') as f:
    f.write(f'PERSON = {PERSON}\nHOSTILE = {HOSTILE}\nA = {A}\nPASSIVE = {PASSIVE}\n'
            f'AGG = {AGG}\nAGG_ACT = {AGG_ACT}\nCOWORKER = {COWORKER}\nACC = {ACC}\n\n'
            'violence = COWORKER or ((A or PASSIVE) and not (ACC and not AGG)) or (AGG and AGG_ACT)\n')

v = df[df.violence]
print(f'violence: {len(v):,} ({len(v)/len(df):.1%})')
print(f'handling: {int(df.handling.sum()):,} (should equal the nursing-facility-handling module: 5,280)')
print(f'violence during handling: {int(df.violence_during_handling.sum()):,} '
      f'({df.violence_during_handling.sum()/len(v):.1%} of violence)')

### Spot check

Twenty random violence narratives. Read them. If the rule misclassifies, fix the rule, not the count.

In [ ]:
for t in v.narr.sample(20, random_state=1): print(' -', t[:220])

---
## 5. Validation against the silver-standard benchmark

Requires `../benchmark/benchmark-500-ai-prelabeled.csv`. The rule was written while reading these cases, so the result is a development score, not an out-of-sample estimate.

In [ ]:
BF = '../benchmark/benchmark-500-ai-prelabeled.csv'
if os.path.exists(BF):
    b = pd.read_csv(BF)
    pred = b.narr.fillna('').str.lower().map(is_violence)
    truth = b.ai_labels.str.contains('violence')
    tp, fp, fn = int((pred & truth).sum()), int((pred & ~truth).sum()), int((~pred & truth).sum())
    val = pd.Series({'sample': len(b), 'violence (silver label)': int(truth.sum()), 'rule positive': int(pred.sum()),
                     'precision': round(tp/max(tp+fp, 1), 3), 'recall': round(tp/max(tp+fn, 1), 3)})
    val.rename('value').to_csv('derived/validation-result.csv')
    print(val.to_string())
    print('\n-- false positives --'); [print(' -', t[:180]) for t in b[pred & ~truth].narr]
    print('\n-- false negatives --'); [print(' -', t[:180]) for t in b[~pred & truth].narr]
else:
    print('benchmark file not found; skipping')

---
## 6. Who is injured

In [ ]:
def occ(r):
    s = f"{r.soc or ''} {r.job or ''}".lower()
    if re.search(r'nursing assistant|\bcna\b|\bstna\b|\bcma\b|aide|orderl', s): return 'Nursing assistant / aide'
    if re.search(r'licensed practical|licensed vocational|\blpn\b|\blvn\b', s): return 'LPN / LVN'
    if re.search(r'registered nurse|\brn\b', s): return 'Registered nurse'
    if re.search(r'therap', s): return 'Therapy'
    if re.search(r'housekeep|janitor|laundry|dietary|cook|food', s): return 'Housekeeping / dietary'
    return 'Other / unspecified'
v = v.assign(occ=v.apply(occ, axis=1))
by_occ = v.occ.value_counts().rename('cases').to_frame().assign(share=lambda d: (d.cases/d.cases.sum()).round(3))
by_occ

---
## 7. Mechanism and body region

Keyword counts; one case can name more than one.

In [ ]:
MECH = {'hit / punch / strike': r'\b(?:hit|struck|strike|punch\w*|slap\w*|swung at)\b', 'kick': r'\bkick\w*',
        'bite': r'\b(?:bit|bite|biting|bitten)\b', 'scratch': r'\bscratch\w*',
        'grab / twist / pinch': r'\b(?:grab\w*|twist\w*|pinch\w*|squeez\w*)', 'push / shove': r'\b(?:push\w*|shov\w*)\b',
        'spit': r'\b(?:spit|spat)\b'}
BODY = {'head / face / eye': r'\b(?:head|face|facial|eye|nose|jaw|mouth|lip|forehead|temple|cheek|ear|concussion)\b',
        'hand / finger / wrist': r'\b(?:hand|finger|thumb|wrist)\b', 'arm / shoulder': r'\b(?:arm|forearm|shoulder|elbow)\b',
        'chest / abdomen': r'\b(?:chest|abdomen|abdominal|stomach|rib|breast)\b', 'back / neck': r'\b(?:back|lumbar|neck|spine)\b',
        'leg / knee / foot': r'\b(?:leg|knee|foot|ankle|shin|thigh)\b'}
tab = lambda D: (pd.Series({k: int(v.narr.str.contains(p, regex=True).sum()) for k, p in D.items()})
                 .rename('cases').to_frame().assign(share=lambda d: (d.cases/len(v)).round(3)))
mech, body = tab(MECH), tab(BODY)
display(mech); display(body)

---
## 8. Severity

In [ ]:
sev = pd.Series({'violence cases': len(v), 'days away (sum)': int(v.dafw.sum()), 'days restricted (sum)': int(v.djtr.sum()),
                 'zero days away': int((v.dafw.fillna(0) == 0).sum()), 'cases > 90 days away': int((v.dafw > 90).sum()),
                 'longest absence': int(v.dafw.max())})
sev

---
## 9. By state and by establishment size

Counts of reported cases. Establishments with no cases are absent from the source file, so these tables do not support rates. Size codes follow the OSHA ITA data dictionary; code 2 was split into 21 and 22 beginning with 2023 data.

In [ ]:
st = (df.groupby('state').agg(cases=('narr', 'size'), establishments=('est', 'nunique'),
                              handling=('handling', 'sum'), violence=('violence', 'sum'))
        .assign(handling_share=lambda d: (d.handling/d.cases).round(3), violence_share=lambda d: (d.violence/d.cases).round(3)))
days = df.groupby('state').apply(lambda g: pd.Series({
    'handling_days_away': int(g[g.handling].dafw.sum()), 'handling_days_restricted': int(g[g.handling].djtr.sum()),
    'violence_days_away': int(g[g.violence].dafw.sum()), 'violence_days_restricted': int(g[g.violence].djtr.sum())}),
    include_groups=False)
st = st.join(days).sort_values('cases', ascending=False)
assert st.cases.sum() == len(df) and st.handling.sum() == df.handling.sum() and st.violence.sum() == df.violence.sum()

SIZE = {'1': 'under 20', '2': '20-249 (pre-2023 code)', '21': '20-99', '22': '100-249', '3': '250+'}
df['size_label'] = df['size'].astype(str).str.strip().map(SIZE).fillna('unknown')
sz = (df.groupby('size_label').agg(cases=('narr', 'size'), establishments=('est', 'nunique'),
                                   handling=('handling', 'sum'), violence=('violence', 'sum'))
        .assign(handling_share=lambda d: (d.handling/d.cases).round(3), violence_share=lambda d: (d.violence/d.cases).round(3))
        .reindex(['under 20', '20-99', '20-249 (pre-2023 code)', '100-249', '250+', 'unknown']).dropna(how='all'))
display(st.head(15)); display(sz)

---
## 10. Export

In [ ]:
by_occ.to_csv('derived/violence-by-occupation.csv')
sev.rename('value').to_csv('derived/violence-severity.csv')
body.to_csv('derived/violence-body-part.csv'); mech.to_csv('derived/violence-mechanism.csv')
st.to_csv('../data/nf-injury-by-state-2025.csv'); sz.to_csv('../data/nf-injury-by-size-2025.csv')
print(sorted(os.listdir('derived')))

---
## Limitations

**Coverage.** Case detail submission is required only of establishments with 100 or more employees in designated industries. Small facilities are under-represented.

**Employer-reported.** Narratives are free text written by employers; detail and wording vary.

**Rule-based classification.** Violence is a keyword rule, not an OSHA field. Its development accuracy is reported in Section 5.

**No rates.** Establishments with no cases do not appear in the case file.

**No establishment is named.** OSHA: recording an injury does not mean the employer was at fault or violated any OSHA rule.